In [1]:
# Save global mortality by year and for each ensemble

In [1]:
import os
import xarray as xr
import numpy as np
import dask
import warnings
from utils.utils import get_scenario_config

In [2]:
def att_frac(x, TMREL, beta):
    O3_diff = x - TMREL
    TMREL_O3 = xr.where(O3_diff > 0, O3_diff, 0)
    RR = xr.apply_ufunc(np.exp, beta * TMREL_O3, dask="allowed")
    AF = (RR - 1) / RR
    return AF


def att_frac2(x, TMREL, beta):
    # equation is: AF = (RR - 1)/RR
    # where RR = e^(beta*(x-TMREL))
    # Akritidis et al. (2024) - unsure which reference would be best

    O3_diff = x - TMREL
    TMREL_O3 = xr.where(O3_diff > 0, O3_diff, 0)  # where the difference < 0 set to 0

    RR = np.exp(beta * TMREL_O3)
    AF = (RR - 1)/RR
    return AF

In [3]:
def calc_PAF(AF, pop, region_mask):
    """
    AF: DataArray(lat, lon, year, samples)
    pop: DataArray(lat, lon)
    region_mask: DataArray(country, lat, lon) with 0/1 mask
    """

    # Numerator: weighted AF summed over lat/lon
    weighted_AF = (AF * pop)                     # (lat, lon, samples)
    numerator = (weighted_AF * region_mask).sum(dim=["lat", "lon"])  # (country, samples)

    # Denominator: population summed over lat/lon
    denominator = (pop * region_mask).sum(dim=["lat", "lon"])        # (country)

    PAF = numerator / denominator
    return PAF

In [4]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

In [22]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load BMR for each country
bmr_file = "GBD_BMR_Country_COPD_newlabels_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [6]:
# === Calculate the scalar distributions (assuming normal dist.) ===
n_samples = 1000

# TMREL from GBD21
tmrel_mean = 32.4
tmrel_std = (35.7 - 29.1) / (2 * 1.96)
tmrel_samples = np.random.normal(tmrel_mean, tmrel_std, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}).astype("float32")

# Beta from RR per 10ppb
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

beta_da = xr.DataArray(
    beta_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}).astype("float32")

In [7]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = [1]  #config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/derecho/scratch/awells/air_quality/{model}/mortality"

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path).astype("float32")
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    # make o3 dask-backed if it isn’t already
    o3 = o3.chunk({'lat': 180, 'lon': 360})

    for year in [2020]:
        print(f"Processing year {year}")
        o3_year = o3.sel(year=year)
        del o3
        AF = att_frac2(o3_year, tmrel_da, beta_da)

        for country in masks.country:
            country_mask = masks.sel(country=country)
            PAF = calc_PAF(AF, pop.sel(year=year), country_mask)
            country_bmr = bmr_da.sel(country=country)
            M = country_bmr * country_mask

        out_file = f"Attributable_fraction_{model}_{scenario}_{ens_num:02d}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        #AF.to_netcdf(out_path)

Processing ensemble member 01
Processing year 2020


In [36]:
country1 = masks.country[0]

In [39]:
test = calc_PAF(AF, pop.sel(year=2020), masks.sel(country=country1))

In [23]:
n_samples = 1000
# BMR from vizhub (GBD21)
bmr_mean = BMR.sel(quantile="mean")
bmr_lower = BMR.sel(quantile="lower")
bmr_upper = BMR.sel(quantile="upper")
bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

bmr_samples = np.random.normal(bmr_mean, bmr_std, size=(n_samples, len(BMR.country)))
bmr_da = xr.DataArray(
    bmr_samples,
    dims=["samples", "country"],
    coords={"samples": np.arange(n_samples),
            "country": BMR.country}
).astype("float32").chunk({"samples": 10})

del bmr_samples

In [42]:
M = bmr_da.sel(country=country1) * test

In [ ]:
M.plot()

In [14]:
population = pop.sel(year=2020).chunk({"lat": 180, "lon": 360})

In [15]:
M = AF.chunk({"samples":10}) * bmr_da * population

In [17]:
global_M = M.sum(dim=("lat", "lon"))

In [20]:
global_M

<xarray.DataArray (samples: 1000)> Size: 8kB
dask.array<sum-aggregate, shape=(1000,), dtype=float64, chunksize=(10,), chunktype=numpy.ndarray>
Coordinates:
  * samples  (samples) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
    year     int64 8B 2020

In [21]:
out_file = f"Global_mortality_{model}_{scenario}_{ens_num:02d}_{year}.nc"
out_path = os.path.join(SAVE_DIR, out_file)
global_M.to_netcdf(out_path)

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = [1]  #config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/test/"

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path)
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    # make o3 dask-backed if it isn’t already
    o3 = o3.chunk({'lat': 180, 'lon': 360, 'year': 1})

    # calculate AF lazily
    AF = att_frac(o3, tmrel_da, beta_da)

    for year in [2030]:  # last year doesn't have OSDMA8
        print(f"Processing year {year}")
        # Calculate population weighted AF and mortality
        PAF = calc_PAF(AF.sel(year=year), pop.sel(year=year), masks)
        M = bmr_da * PAF

        # mean over samples
        M_mean = M.mean(dim="samples").chunk(-1)
        # percentiles for 95% CI
        M_lower = M.quantile(0.025, dim="samples").chunk(-1)
        M_upper = M.quantile(0.975, dim="samples").chunk(-1)

        M_summary = xr.Dataset({
            "M_mean": M_mean.astype("float32").drop_vars("year"),
            "M_lower95": M_lower.astype("float32").drop_vars("quantile"),
            "M_upper95": M_upper.astype("float32").drop_vars("quantile")
        })

        #save_file = os.path.join(
        #    SAVE_DIR,
        #    f"mortality_{model}_{scenario}_{ens_num:02d}_{year}.zarr"
        #)

        # Lazy write
        #delayed = M_summary.to_zarr(save_file)
        #dask.compute(delayed)  # executes, but chunkwise
        #del M_summary, M_mean, M_lower, M_upper

In [ ]:
M_summary.nbytes

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = [1]  #config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/"

# Define decade chunks
years_array = np.arange(years.start, years.stop)  # e.g. 2015–2084
decade_edges = np.arange(years.start, years.stop, 10)  # step = 10 years

In [ ]:
for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path)
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    o3 = o3.chunk({'lat': 135, 'lon': 360, 'year': 64})

    # calculate AF lazily
    AF = att_frac(o3, tmrel_da, beta_da)

    for start in decade_edges:
        stop = min(start + 10, years.stop)  # last block may be <10 years
        block_years = np.arange(start, stop)
        print(f"Processing years {start}-{stop-1}")

        block_results = []
        for year in block_years:
            # Calculate population weighted AF and mortality
            PAF = calc_PAF(AF.sel(year=year), pop.sel(year=year), masks)
            M = bmr_da * PAF

            # mean over samples
            M_mean = M.mean(dim="samples")
            # percentiles for 95% CI
            M_lower = M.quantile(0.025, dim="samples")
            M_upper = M.quantile(0.975, dim="samples")

            M_summary = xr.Dataset({
                "M_mean": M_mean.drop_vars("year"),
                "M_lower95": M_lower.drop_vars("quantile"),
                "M_upper95": M_upper.drop_vars("quantile")
            })
            block_results.append(M_summary)

        # Concatenate within the block
        block_ds = xr.concat(block_results, dim="year")

        # Save each block separately — avoids memory blow-up
        save_file = f"TESTmortality_{model}_{scenario}_{ens_num:02d}_{start}-{stop-1}.nc"
        block_ds.to_netcdf(os.path.join(SAVE_DIR, save_file))
        del block_ds, block_results